# V4 — DINOv3 + k-NN / Linear Probe
## Plant Disease Detection — Self-supervised Foundation Model with frozen backbone

**Paradigm:** DINOv3 (Meta AI) is a Vision Transformer pre-trained **self-supervised** on LVD-1689M without labels.  
We use it as a *frozen feature extractor*: no gradients, no fine-tuning.  
On top we put two very lightweight classifiers:
1. **k-NN** (cosine, k=20) on the CLS vectors — standard SSL evaluation protocol
2. **Linear probe** (logistic regression) on the same vectors — single trainable layer

**Contrast with V3:** V3 = **supervised** pre-training (ImageNet) + **fine-tuning** of layer4.  
V4 = **self-supervised** pre-training (no labels) + **zero training** of the backbone.

In [ ]:
# ── Cell 1: Setup & Imports ─────────────────────────────
import sys, os, json, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from transformers import AutoModel, AutoImageProcessor

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report)

# ── Paths ────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve()
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR  = PROJECT_ROOT / "results"
MODELS_DIR   = RESULTS_DIR / "models" / "v4_dinov3_probe"
EMB_DIR      = MODELS_DIR / "embeddings"
METRICS_DIR  = RESULTS_DIR / "metrics"
PLOTS_DIR    = RESULTS_DIR / "plots"

for d in [MODELS_DIR, EMB_DIR, METRICS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# ── Backbone identifier ────────────────────────────────
# DINOv3 ViT-B/16 pre-trained on LVD-1689M (Meta AI, self-supervised, no labels)
# Gated repo: requires `huggingface-cli login` + accepting the terms on the model card.
# Non-gated fallback: "facebook/dinov2-base" (LVD-142M, ViT-B/14, same paradigm).
MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"

# Slug used to name the embedding caches (so V2 and V3 don't get mixed up)
BACKBONE_SLUG = MODEL_ID.split("/")[-1].replace(".", "_")

# ── Hyperparameters ──────────────────────────────────
IMG_SIZE       = 224
BATCH_SIZE     = 64
NUM_WORKERS    = 0
KNN_K          = 20
KNN_METRIC     = "cosine"
LINPROBE_C     = 1.0
LINPROBE_ITERS = 1000

FAST_MODE = False
print(f"Fast mode: {FAST_MODE}")
print(f"Backbone:  {MODEL_ID}")
print(f"Cache tag: {BACKBONE_SLUG}")

In [ ]:
# ── Cell 2: Dataset & DataLoaders (NO augmentation: deterministic features) ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Same transform for train/val/test: we want stable features, no data aug
extract_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=extract_transform)
val_dataset   = datasets.ImageFolder(DATA_DIR / "val",   transform=extract_transform)
test_dataset  = datasets.ImageFolder(DATA_DIR / "test",  transform=extract_transform)

if FAST_MODE:
    def subsample(ds, frac=0.1):
        n = max(1, int(len(ds) * frac))
        idx = torch.randperm(len(ds))[:n].tolist()
        return torch.utils.data.Subset(ds, idx)
    train_dataset = subsample(train_dataset)
    val_dataset   = subsample(val_dataset)
    test_dataset  = subsample(test_dataset)

# shuffle=False: deterministic order → embedding↔label alignment stays consistent
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

CLASS_NAMES = train_dataset.classes if hasattr(train_dataset, "classes") else train_dataset.dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Classes: {NUM_CLASSES}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# ── Cell 3: Load DINOv3 backbone (frozen) ─────────────────────────
print(f"Loading backbone '{MODEL_ID}'...")
backbone = AutoModel.from_pretrained(MODEL_ID)
backbone.eval()

# Freeze EVERYTHING — we don't update anything
for p in backbone.parameters():
    p.requires_grad = False

backbone = backbone.to(DEVICE)

total_params     = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"Total backbone parameters:     {total_params:,}  (~{total_params/1e6:.1f}M)")
print(f"Trainable backbone parameters: {trainable_params:,}  (all frozen ✅)")

# Probe the embedding dimension with a dummy forward pass
with torch.no_grad():
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out   = backbone(pixel_values=dummy)
    # DINOv3/v2: we use the CLS token = last_hidden_state[:, 0, :]
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        feat_dim = out.pooler_output.shape[-1]
        FEAT_SRC = "pooler_output"
    else:
        feat_dim = out.last_hidden_state.shape[-1]
        FEAT_SRC = "last_hidden_state[:,0]"
print(f"Embedding dim: {feat_dim}  (source: {FEAT_SRC})")

In [ ]:
# ── Cell 4: Extract embeddings with on-disk caching ──────────────────
@torch.no_grad()
def extract_embeddings(loader, backbone, device, feat_src):
    feats, labels = [], []
    for images, lbls in tqdm(loader, desc="Extracting", unit="batch"):
        images = images.to(device, non_blocking=True)
        out    = backbone(pixel_values=images)
        if feat_src == "pooler_output":
            z = out.pooler_output
        else:
            z = out.last_hidden_state[:, 0, :]  # CLS token
        feats.append(z.cpu().numpy().astype(np.float32))
        labels.append(lbls.numpy())
    return np.concatenate(feats), np.concatenate(labels)


def get_or_extract(split_name, loader):
    """Load embeddings from cache if they exist, otherwise extract+save.
    The cache name includes BACKBONE_SLUG: V2 and V3 don't overwrite each other."""
    cache = EMB_DIR / f"{split_name}__{BACKBONE_SLUG}.npz"
    if cache.exists():
        data = np.load(cache)
        print(f"  [{split_name}] cache hit ({cache.name}) → X={data['X'].shape}, y={data['y'].shape}")
        return data["X"], data["y"]
    t0 = time.time()
    X, y = extract_embeddings(loader, backbone, DEVICE, FEAT_SRC)
    dt = time.time() - t0
    np.savez_compressed(cache, X=X, y=y)
    print(f"  [{split_name}] extracted ({cache.name}) → X={X.shape}, y={y.shape}  ({dt/60:.1f} min)")
    return X, y


print(f"Extracting embeddings with backbone '{MODEL_ID}' (cached on disk):")
t_all = time.time()
X_train, y_train = get_or_extract("train", train_loader)
X_val,   y_val   = get_or_extract("val",   val_loader)
X_test,  y_test  = get_or_extract("test",  test_loader)
extract_time_min = (time.time() - t_all) / 60

print(f"\nEmbedding shape:  X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")
print(f"Total extraction time (cache included): {extract_time_min:.1f} min")

In [ ]:
# ── Cell 5: k-NN Classifier (cosine, k=20) ─────────────────────────
print(f"k-NN: k={KNN_K}, metric={KNN_METRIC}")

t0 = time.time()
knn = KNeighborsClassifier(n_neighbors=KNN_K, metric=KNN_METRIC, n_jobs=-1)
knn.fit(X_train, y_train)
knn_fit_time = time.time() - t0

t0 = time.time()
preds_knn = knn.predict(X_test)
knn_pred_time = time.time() - t0

acc_knn  = accuracy_score(y_test, preds_knn)
prec_knn = precision_score(y_test, preds_knn, average="weighted", zero_division=0)
rec_knn  = recall_score(y_test, preds_knn, average="weighted", zero_division=0)
f1_knn   = f1_score(y_test, preds_knn, average="weighted", zero_division=0)

print(f"\n── V4 k-NN Metrics ─────────────────")
print(f"  accuracy:  {acc_knn:.4f}")
print(f"  precision: {prec_knn:.4f}")
print(f"  recall:    {rec_knn:.4f}")
print(f"  f1:        {f1_knn:.4f}")
print(f"  fit time:  {knn_fit_time:.1f}s | pred time: {knn_pred_time:.1f}s")

In [ ]:
# ── Cell 6: Linear Probe (Logistic Regression) ─────────────────────
# L2 normalization on the embeddings: standard protocol for SSL linear probe
X_train_n = normalize(X_train, norm="l2")
X_test_n  = normalize(X_test,  norm="l2")

print(f"Linear probe: LogisticRegression(C={LINPROBE_C}, max_iter={LINPROBE_ITERS})")

t0 = time.time()
linprobe = LogisticRegression(
    C=LINPROBE_C,
    max_iter=LINPROBE_ITERS,
    n_jobs=-1,
    solver="lbfgs",
)
linprobe.fit(X_train_n, y_train)
lin_fit_time = time.time() - t0

t0 = time.time()
preds_lin = linprobe.predict(X_test_n)
lin_pred_time = time.time() - t0

acc_lin  = accuracy_score(y_test, preds_lin)
prec_lin = precision_score(y_test, preds_lin, average="weighted", zero_division=0)
rec_lin  = recall_score(y_test, preds_lin, average="weighted", zero_division=0)
f1_lin   = f1_score(y_test, preds_lin, average="weighted", zero_division=0)

print(f"\n── V4 Linear Probe Metrics ──────────")
print(f"  accuracy:  {acc_lin:.4f}")
print(f"  precision: {prec_lin:.4f}")
print(f"  recall:    {rec_lin:.4f}")
print(f"  f1:        {f1_lin:.4f}")
print(f"  fit time:  {lin_fit_time:.1f}s | pred time: {lin_pred_time:.1f}s")

# ── Save unified metrics ──────────────────────────────────
metrics = {
    "model":      "V4_DINOv3_FrozenBackbone",
    "backbone":   MODEL_ID,
    "embedding_dim": int(X_train.shape[1]),
    "extract_time_min": round(extract_time_min, 2),
    "backbone_params":   int(total_params),
    "trainable_params":  int(trainable_params),  # 0 for the backbone
    "knn": {
        "k": KNN_K, "metric": KNN_METRIC,
        "accuracy":  round(acc_knn, 4),
        "precision": round(prec_knn, 4),
        "recall":    round(rec_knn, 4),
        "f1":        round(f1_knn, 4),
        "fit_time_s":  round(knn_fit_time, 2),
        "pred_time_s": round(knn_pred_time, 2),
    },
    "linear_probe": {
        "C": LINPROBE_C, "max_iter": LINPROBE_ITERS,
        "accuracy":  round(acc_lin, 4),
        "precision": round(prec_lin, 4),
        "recall":    round(rec_lin, 4),
        "f1":        round(f1_lin, 4),
        "fit_time_s":  round(lin_fit_time, 2),
        "pred_time_s": round(lin_pred_time, 2),
    },
    # For compatibility with the comparison table, V4's "best" is the linear probe:
    "accuracy":  round(max(acc_knn, acc_lin), 4),
    "precision": round(prec_lin if acc_lin >= acc_knn else prec_knn, 4),
    "recall":    round(rec_lin  if acc_lin >= acc_knn else rec_knn,  4),
    "f1":        round(f1_lin   if acc_lin >= acc_knn else f1_knn,   4),
    "best_classifier": "linear_probe" if acc_lin >= acc_knn else "knn",
}
with open(METRICS_DIR / "v4_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nSaved to results/metrics/v4_metrics.json ✅")

In [ ]:
# ── Cell 7: Confusion Matrix + Comparison plot k-NN vs Linear Probe ──
# Confusion matrix of the best classifier
best_preds = preds_lin if acc_lin >= acc_knn else preds_knn
best_acc   = max(acc_lin, acc_knn)
best_name  = "Linear Probe" if acc_lin >= acc_knn else "k-NN"

cm = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=False, fmt="d", cmap="Purples",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title(f"V4 DINOv3 + {best_name} — Confusion Matrix\nAccuracy: {best_acc:.4f}", fontsize=14)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
out_path = PLOTS_DIR / "v4_confusion_matrix.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path} ✅")

# Bar chart comparing k-NN vs Linear Probe on the 4 metrics
labels  = ["Accuracy", "Precision", "Recall", "F1"]
vals_kn = [acc_knn, prec_knn, rec_knn, f1_knn]
vals_lp = [acc_lin, prec_lin, rec_lin, f1_lin]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, vals_kn, w, label=f"k-NN (k={KNN_K})",      color="steelblue")
b2 = ax.bar(x + w/2, vals_lp, w, label="Linear Probe", color="seagreen")
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.3f}", xy=(bar.get_x()+bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("V4 DINOv3 — k-NN vs Linear Probe")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
out_path2 = PLOTS_DIR / "v4_knn_vs_linprobe.png"
plt.savefig(out_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path2} ✅")

# Per-class F1 (best classifier)
f1_per_class = f1_score(y_test, best_preds, average=None, zero_division=0)
fig2, ax2 = plt.subplots(figsize=(18, 5))
ax2.bar(CLASS_NAMES, f1_per_class, color="mediumpurple")
ax2.set_xticks(range(len(CLASS_NAMES)))
ax2.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=7)
ax2.set_ylabel("F1 Score")
ax2.set_title(f"V4 DINOv3 + {best_name} — Per-class F1")
ax2.set_ylim(0, 1.05)
ax2.axhline(f1_per_class.mean(), color="red", linestyle="--", label=f"Mean: {f1_per_class.mean():.3f}")
ax2.legend()
plt.tight_layout()
out_path3 = PLOTS_DIR / "v4_f1_per_class.png"
plt.savefig(out_path3, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path3} ✅")
print("\nDone! Notebook 04 (V4 DINOv3) complete.")